# Demostración de Estimación mediante Evolución Diferencial en Modelos de Difusión con Saltos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdospina/jump-diffusion-estimation/blob/main/notebooks/differential_evolution_showcase_spanish.ipynb)

Este notebook ilustra la aplicación del algoritmo de **Evolución Diferencial (optimización global)** frente al algoritmo clásico **L-BFGS-B (optimización local)** en la estimación de parámetros para modelos de difusión con saltos.

## El Problema: Verosimilitudes No Convexas y Multimodales

En los modelos de difusión con saltos (como el modelo de Merton o extensiones asimétricas como Kou y SGED), la función de verosimilitud se construye como una mixtura infinita (o aproximada) de densidades de probabilidad.

Cuando se utiliza una distribución de saltos flexible y pesada como la **Distribución de Error Generalizada Asimétrica (SGED)**, la superficie de la log-verosimilitud resultante presenta:
1. Múltiples óptimos locales (multimodalidad).
2. Regiones planas o crestas estrechas donde los gradientes son casi nulos.
3. Alta sensibilidad a los valores iniciales.

Como se demostró en la tesis de *Ospina Arango (2009)*, los optimizadores basados en gradientes locales como **L-BFGS-B** a menudo se estancan en óptimos locales deficientes o en las fronteras de los parámetros, mientras que la **Evolución Diferencial (DE)**, al ser un método de búsqueda global estocástico y libre de derivadas, es capaz de recuperar con gran precisión los parámetros verdaderos sin requerir una aproximación inicial precisa.

## 1. Configuración del Entorno e Importaciones

Primero importamos las librerías necesarias para la simulación, estimación, análisis estadístico y graficación.

In [ ]:
# Instalación de dependencias necesarias
%pip install -q jump-diffusion-estimation ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

from jump_diffusion import JumpDiffusionSimulator, JumpDiffusionEstimator
from jump_diffusion.distributions import SGEDJump

# Configuración de estilos visuales para los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 2. Simulación de Datos Sintéticos con Parámetros Conocidos

Definimos un conjunto de parámetros verdaderos y simulamos una trayectoria diaria durante 3 años ($T = 3.0$ años, $n\_steps = 756$ observaciones diarias). Utilizaremos la distribución **SGED** para el tamaño de los saltos, configurándola con asimetría positiva sustancial ($\xi = 1.8$) y colas pesadas ($\nu = 1.3$).

In [ ]:
# Parámetros reales del modelo
true_params = {
    "mu": 0.12,          # 12% deriva anual
    "sigma": 0.25,       # 25% volatilidad de difusión
    "jump_prob": 0.08,   # 8% probabilidad de salto diaria
}

# Parámetros específicos de los saltos (SGED)
true_jump_params = {
    "jump_loc": 0.0,     # centrado en cero
    "jump_scale": 0.15,  # escala del tamaño de salto
    "jump_nu": 1.3,      # forma de las colas (< 2.0 implica colas más pesadas que la normal)
    "jump_xi": 1.8,      # asimetría positiva (> 1.0 implica asimetría positiva)
}

# Instanciar simulador
simulador = JumpDiffusionSimulator(
    jump_distribution=SGEDJump(),
    **true_params,
    **true_jump_params
)

# Simular la trayectoria
T = 3.0
n_steps = 756
times, path, jumps = simulador.simulate_path(T=T, n_steps=n_steps, x0=100.0, seed=42)
increments = np.diff(path)
dt = T / n_steps

print(f"Se simularon {len(path)} puntos de trayectoria con {np.sum(jumps != 0)} saltos realizados.")

In [ ]:
# Graficar la trayectoria simulada y sus componentes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(times, path, color="royalblue", label="Trayectoria X(t)")
jump_times = times[1:][jumps != 0]
jump_vals = path[1:][jumps != 0]
ax1.scatter(jump_times, jump_vals, color="crimson", zorder=5, label="Saltos Realizados")
ax1.set_title("Trayectoria Simulada de Difusión con Saltos")
ax1.set_ylabel("Precio / Valor")
ax1.legend()

ax2.stem(times[1:], jumps, linefmt="crimson", markerfmt="o", basefmt=" ", label="Magnitud del salto")
ax2.set_ylabel("Magnitud del Salto")
ax2.set_xlabel("Tiempo (Años)")
ax2.set_title("Instantes y Magnitudes de los Saltos")
plt.tight_layout()
plt.show()

## 3. Estimación mediante Optimización Local (L-BFGS-B)

Intentaremos estimar los parámetros usando `L-BFGS-B`. Este algoritmo requiere una suposición inicial de los parámetros. La librería calcula de forma predeterminada una estimación basada en los momentos estadísticos de la muestra (medias, varianzas y asimetrías de los incrementos) para dar una buena aproximación de arranque.

In [ ]:
estimator_lbfgsb = JumpDiffusionEstimator(increments, dt, jump_distribution=SGEDJump())

t_start = time.time()
results_lbfgsb = estimator_lbfgsb.estimate(method="L-BFGS-B")
t_lbfgsb = time.time() - t_start

print(f"Tiempo de ejecución de L-BFGS-B: {t_lbfgsb:.4f} segundos.")
estimator_lbfgsb.diagnostics()

## 4. Estimación mediante Optimización Global (Evolución Diferencial)

Ahora ejecutamos el estimador configurado para usar `differential_evolution`. Este algoritmo no requiere de una suposición inicial de parámetros (genera una población uniforme dentro de un hipercubo de búsqueda acotado y adaptativo derivado de los datos). El costo computacional es notablemente mayor (miles de evaluaciones de la verosimilitud de la mixtura), pero es mucho más inmune a los óptimos locales.

In [ ]:
estimator_de = JumpDiffusionEstimator(increments, dt, jump_distribution=SGEDJump())

t_start = time.time()
# Usamos una semilla fija (seed=42) para reproducibilidad
results_de = estimator_de.estimate(method="differential_evolution", seed=42)
t_de = time.time() - t_start

print(f"Tiempo de ejecución de Evolución Diferencial: {t_de:.4f} segundos (evaluando ~4000-8000 candidatos).")
estimator_de.diagnostics()

## 5. Comparación Numérica y Gráfica

Comparemos las estimaciones arrojadas por ambos algoritmos frente a los parámetros verdaderos que utilizamos para simular el proceso.

In [ ]:
params_verdaderos = {**true_params, **true_jump_params}
params_lbfgsb = results_lbfgsb["parameters"]
params_de = results_de["parameters"]

df_comp = pd.DataFrame({
    "Parámetro": list(params_verdaderos.keys()),
    "Valor Real": list(params_verdaderos.values()),
    "L-BFGS-B (Local)": [params_lbfgsb[k] for k in params_verdaderos.keys()],
    "Evol. Diferencial (Global)": [params_de[k] for k in params_verdaderos.keys()]
})

df_comp["Error Rel. L-BFGS-B (%)"] = np.abs((df_comp["L-BFGS-B (Local)"] - df_comp["Valor Real"]) / np.maximum(df_comp["Valor Real"], 1e-5)) * 100
df_comp["Error Rel. DE (%)"] = np.abs((df_comp["Evol. Diferencial (Global)"] - df_comp["Valor Real"]) / np.maximum(df_comp["Valor Real"], 1e-5)) * 100

df_comp.round(4)

### Log-Verosimilitudes Comparadas

Una métrica fundamental para evaluar el éxito de la optimización es la log-verosimilitud final alcanzada. Dado que estamos *maximizando* esta verosimilitud, el algoritmo que obtenga un valor más alto habrá encontrado un mejor ajuste de los datos.

In [ ]:
ll_real = estimator_de.log_likelihood(np.array(list(params_verdaderos.values()))) * -1
ll_lbfgsb = results_lbfgsb["log_likelihood"]
ll_de = results_de["log_likelihood"]

print(f"Log-Verosimilitud teórica (parámetros reales):     {ll_real:.2f}")
print(f"Log-Verosimilitud alcanzada por L-BFGS-B:            {ll_lbfgsb:.2f}")
print(f"Log-Verosimilitud alcanzada por Evolución Diferencial: {ll_de:.2f}")

diferencia = ll_de - ll_lbfgsb
print(f"\nDiferencia a favor de Evolución Diferencial: {diferencia:.2f} unidades de log-verosimilitud.")
if diferencia > 1.0:
    print("¡DE superó significativamente a L-BFGS-B, demostrando que el optimizador local quedó atrapado en un óptimo local!")
else:
    print("Ambos optimizadores llegaron a zonas similares en verosimilitud.")

### Comparación Gráfica de Ajuste de Densidades

Para evaluar visualmente los ajustes, simulamos muestras sintéticas de incrementos utilizando los parámetros estimados por ambos optimizadores y las comparamos mediante histogramas superpuestos con los datos reales.

In [ ]:
from jump_diffusion.models import JumpDiffusionModel

# Simular muestras sintéticas para verificar la bondad de ajuste
model_lbfgsb = JumpDiffusionModel(jump_distribution=SGEDJump(), **params_lbfgsb)
model_de = JumpDiffusionModel(jump_distribution=SGEDJump(), **params_de)

_, path_lbfgsb, _ = model_lbfgsb.simulate(T=T, n_steps=n_steps, x0=0.0, seed=100)
_, path_de, _ = model_de.simulate(T=T, n_steps=n_steps, x0=0.0, seed=100)

inc_lbfgsb = np.diff(path_lbfgsb)
inc_de = np.diff(path_de)

# Gráfico de densidades comparadas
plt.figure(figsize=(14, 7))
sns.kdeplot(increments, fill=True, color="black", alpha=0.15, linewidth=2.5, label="Datos Reales (Simulados)")
sns.kdeplot(inc_lbfgsb, color="crimson", linestyle="--", linewidth=2.0, label=f"Ajuste L-BFGS-B (Log-Lik: {ll_lbfgsb:.1f})")
sns.kdeplot(inc_de, color="forestgreen", linestyle="-", linewidth=2.0, label=f"Ajuste Evol. Diferencial (Log-Lik: {ll_de:.1f})")

plt.title("Ajuste Empírico de Densidades de Incrementos")
plt.xlabel("Incremento (\Delta X)")
plt.ylabel("Densidad")
plt.legend(frameon=True, facecolor="white")
plt.xlim(-0.35, 0.45)
plt.show()

## 6. Conclusiones y Discusión

Este experimento de validación muestra claramente los contrastes de rendimiento entre algoritmos locales y globales en la estimación de mixturas complejas:

1. **Estancamiento Local**: `L-BFGS-B` se ejecuta de forma extremadamente rápida (milisegundos) pero suele converger a estimaciones erróneas en los parámetros de la distribución de saltos (`jump_nu` y `jump_xi`). Esto se debe a que la verosimilitud de la mixtura es multimodal y el punto de partida (guess de momentos) no siempre cae en la cuenca de atracción del máximo global.
2. **Precisión del Máximo Global**: `differential_evolution` explora sistemáticamente todo el espacio de búsqueda. Aunque tarda más en ejecutarse (en el rango de segundos), recupera con precisión milimétrica los parámetros verdaderos de deriva, volatilidad, probabilidad y asimetría de los saltos. Su log-verosimilitud final es consistentemente más alta que la de L-BFGS-B.
3. **Uso Recomendado**: En entornos de producción o investigación académica (donde la precisión de la estimación es crítica para la valoración de derivados o gestión de riesgos), la **Evolución Diferencial** es la metodología de estimación predeterminada recomendada para modelos de mixturas de saltos asimétricas y con colas pesadas.

## Cálculo de Errores Estándar por Perfilado de Verosimilitud

Una vez que hemos estimado los parámetros (preferiblemente usando Evolución Diferencial para asegurar el óptimo global), podemos calcular los errores estándar y los intervalos de confianza (95%) utilizando la técnica de **Perfilado de la Log-Verosimilitud** (Likelihood Profiling).

Esta técnica es mucho más robusta que el cálculo numérico de la matriz Hessiana (la cual suele ser inestable en mixturas de distribuciones complejas).

In [ ]:
# Estimar errores estándar mediante perfilado (utilizaremos n_points=5 para mayor rapidez en este ejemplo)
# Asegúrate de haber ejecutado estimate() previamente en el estimador.
try:
    # Intentamos usar estimator_de si existe (del showcase de Evolución Diferencial)
    est = estimator_de
except NameError:
    try:
        # Intentamos usar estimator si existe (del ejemplo general)
        est = estimator
    except NameError:
        print("Asegúrate de definir un 'estimator' y ejecutar estimate() primero.")
        est = None

if est is not None:
    print("Calculando errores estándar...")
    se_results = est.estimate_standard_errors(n_points=5, confidence_level=0.95)
    
    # Imprimir los diagnósticos actualizados que ahora incluyen la tabla de Errores Estándar e Intervalos
    est.diagnostics()
    
    # Graficar las curvas de verosimilitud perfilada para cada parámetro
    est.plot_profiles()
